# 03. Как превратить текст в числа

В этом блокноте мы подготовим текст для нейросети. Пока не строим Transformer: учимся превращать предложения в числовые последовательности.

## Что изучим
1. Что такое токен и токенизация.
2. Как построить словарь (vocabulary).
3. Как заменить слова числовыми ID.
4. Зачем нужны специальные токены `<PAD>` и `<UNK>`.
5. Как сделать последовательности одинаковой длины и собрать batch.

**Важно:** наш токенизатор учебный и очень простой. В реальной NLP-практике используют более продвинутые методы. Мы начнём с простого, чтобы понимать каждый шаг.

## 1. Термины

- **Токен (token)** — единица текста, которую модель получает на вход. В этом блокноте токеном будет слово.
- **Токенизация (tokenization)** — разбиение текста на токены.
- **Словарь (vocabulary)** — набор токенов, которым присвоены числовые ID.
- **Token ID** — целое число, обозначающее конкретный токен в словаре.
- **Padding** — добавление специальных элементов, чтобы все последовательности в одном batch имели одинаковую длину.
- **Batch (батч)** — группа примеров, которую модель обрабатывает за один шаг.


## 2. Токенизация: разбиваем предложение на слова

Начнём с метода Python `split()`. Он разбивает строку по пробелам. Метод `lower()` переводит буквы в нижний регистр, чтобы `Movie` и `movie` не считались разными словами.

Это упрощение: пока знаки препинания остаются частью токенов. Позже обсудим, почему это неидеально.

In [ ]:
text = "The movie was surprisingly good."

normalized_text = text.lower()
tokens = normalized_text.split()

print("Исходный текст:", text)
print("После lower():", normalized_text)
print("Токены:", tokens)
print("Количество токенов:", len(tokens))

### Что здесь произошло?

`text` — переменная, в которой хранится строка.
`lower()` возвращает строку в нижнем регистре.
`split()` возвращает список слов.
`len(tokens)` считает количество элементов списка.

Обрати внимание: точка осталась прикреплена к слову `good.`. Для первого знакомства это допустимо, но в дальнейшем мы улучшим обработку текста.

## 3. Создаём словарь

Модель не может использовать слова напрямую в матричных вычислениях. Поэтому каждому уникальному токену присвоим целое число.

Сначала возьмём несколько предложений. В реальном проекте словарь будет строиться по обучающей части датасета, а не по проверочным данным.

In [ ]:
texts = [
    "The movie was good",
    "The movie was bad",
    "I really liked the movie",
    "I did not like the movie"
]

tokenized_texts = [text.lower().split() for text in texts]

vocab = {"<PAD>": 0, "<UNK>": 1}

for tokens in tokenized_texts:
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

print("Словарь:")
print(vocab)
print("Размер словаря:", len(vocab))

### Зачем нужны `<PAD>` и `<UNK>`?

- `<PAD>` — специальный токен-заполнитель. Им будем выравнивать длину предложений.
- `<UNK>` (unknown) — специальный токен для слова, которого нет в словаре.

Мы заранее закрепили за ними ID 0 и 1. Это важно: специальные ID должны иметь постоянное значение.

## 4. Кодируем текст: слова → числа

Теперь создадим обратный словарь `id_to_token`, чтобы при необходимости переводить ID обратно в слова.
Затем напишем функцию `encode`, которая заменяет каждый токен его числовым ID.

In [ ]:
id_to_token = {token_id: token for token, token_id in vocab.items()}

def encode(text, vocab):
    tokens = text.lower().split()
    token_ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    return token_ids

example = "The movie was amazing"
encoded = encode(example, vocab)

print("Текст:", example)
print("ID:", encoded)
print("Обратное преобразование:", [id_to_token.get(i, "<неизвестный ID>") for i in encoded])

### Разберём новую конструкцию

`def encode(...)` — определение функции. Функция получает текст и словарь, выполняет инструкции и возвращает список ID.

`vocab.get(token, vocab["<UNK>"])` означает: найди ID токена; если токена нет, верни ID `<UNK>`.

В примере слово `amazing` отсутствует в нашем маленьком словаре. Поэтому оно превращается в `<UNK>`.

## 5. Padding: выравниваем длину предложений

Предложения имеют разную длину. Но для удобной пакетной обработки мы хотим собрать их в прямоугольную матрицу: одинаковое число столбцов у каждого предложения.

Короткие последовательности дополним ID токена `<PAD>` (у нас это 0). Длинные при необходимости обрежем до `max_length`.

In [ ]:
def encode_and_pad(text, vocab, max_length):
    token_ids = encode(text, vocab)
    token_ids = token_ids[:max_length]  # обрезаем слишком длинную последовательность
    padding_length = max_length - len(token_ids)
    token_ids += [vocab["<PAD>"]] * padding_length
    return token_ids

max_length = 6
encoded_texts = [encode_and_pad(text, vocab, max_length) for text in texts]

for text, ids in zip(texts, encoded_texts):
    print(text)
    print(ids)
    print()

### Attention mask — маска настоящих токенов

Позже Transformer должен отличать реальные слова от добавленных `<PAD>`. Для этого создают маску: 1 означает настоящий токен, 0 — padding.

Пока просто построим её. Подробно разберём, как маска используется внутри Attention, когда дойдём до этой части архитектуры.

In [ ]:
import torch

input_ids = torch.tensor(encoded_texts, dtype=torch.long)
attention_mask = (input_ids != vocab["<PAD>"]).long()

print("Матрица ID:")
print(input_ids)
print("Форма матрицы:", input_ids.shape)

print("\nМаска:")
print(attention_mask)
print("Форма маски:", attention_mask.shape)

## 6. Что такое batch?

В предыдущей ячейке мы получили матрицу. Каждая строка — отдельное предложение, каждый столбец — позиция токена.

В нашем примере форма `(4, 6)` означает:
- 4 предложения в одном batch;
- 6 позиций токенов в каждом предложении.

Именно такой формат входных данных часто используется при обучении Transformer: `(batch_size, sequence_length)`.

## 7. Самостоятельное упражнение

Не переходи дальше, пока не попробуешь решить самостоятельно.

1. Добавь в `texts` предложение `The movie was fantastic` и заново построй словарь.
2. Закодируй предложение `I loved this film`.
3. Объясни, почему часть слов может превратиться в `<UNK>`.
4. Измени `max_length` с 6 на 8. Что изменится в `input_ids.shape`?
5. Объясни, почему `<PAD>` и `<UNK>` имеют разные ID и выполняют разные функции.

**Важно:** если меняешь `texts` и словарь, перезапусти ячейки ниже по порядку, чтобы все переменные соответствовали новым данным.

## Что мы узнали

Мы научились превращать текст в числовые ID, выравнивать длины последовательностей и собирать предложения в матрицу.

Следующий шаг — **Embedding**: как заменить каждый ID не одним числом, а целым обучаемым вектором. Именно эти векторы станут входом Transformer.